# **Многомерный дисперсионный анализ (MANOVA): Комплексная оценка различий между кластерами**

* __Цель многомерного дисперсионного анализа:__ Определить, существуют ли статистически значимые многомерные различия между выделенными клиническими профилями по всей совокупности количественных медицинских признаков одновременно, доказав уникальность центроидов групп и минимизировав вероятность ложноположительных срабатываний (ошибки I рода).
* __Задачи многомерного дисперсионного анализа:__ Оценить совместное влияние категориальных факторов на вектор зависимых переменных. Математически подтвердить разделительную способность выбранных алгоритмов машинного обучения. Выявить конкретные физиологические маркеры (драйверы), вносящие наибольший вклад в сегментацию, и доказать отсутствие статистического смещения при перекрестном взаимодействии признаков.
* __Алгоритм использования:__
  1. __Диагностика данных:__ Проверка строгих математических допущений для применения MANOVA (диагностика многомерной нормальности через расстояние Махаланобиса и оценка гомогенности ковариационных матриц критерием Бокса) для выбора наиболее робастной тестовой статистики.
  2. __Однофакторный анализ (One-Way MANOVA) и Post-Hoc:__ Выдвижение многомерных гипотез ($H_0$ и $H_1$) и глобальная оценка центроидов. При выявлении статистической значимости — проведение апостериорного анализа (одномерные ANOVA с жесткой поправкой Бонферрони и критерий Тьюки) для точной локализации различий.
  3. __Двухфакторный анализ (Two-Way MANOVA):__ Оценка одновременного влияния клинического профиля и дополнительного демографического фактора (пола). Проверка значимости эффекта взаимодействия для доказательства универсальности модели.
  4. __Математический отбор и визуализация (Model Selection & LDA):__ Ранжирование алгоритмов кластеризации (K-Means, DBSCAN, Hierarchical) по величине следа Пиллая (Pillai's trace). Проекция многомерного пространства алгоритма-победителя в интерпретируемую плоскость с помощью Линейного дискриминантного анализа (LDA).

In [ ]:
# --- ИМПОРТ БИБЛИОТЕК И ПЕРВИЧНАЯ НАСТРОЙКА ---

# Сторонние библиотеки
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pingouin as pg
import seaborn as sns
import statsmodels.api as sm
from IPython.display import Markdown, display
from scipy import stats
from scipy.spatial import distance
from statsmodels.formula.api import ols
from statsmodels.multivariate.manova import MANOVA
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Локальные модули
from scripts.logger import setup_logger

# Инициализация логгера
logger = setup_logger("manova")

# Установка уровня логирования для предупреждений от matplotlib
logging.getLogger("matplotlib.category").setLevel(logging.WARNING)

# Настройка визуализации
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

# Константы
ALPHA_LEVEL = 0.05  # Уровень статистической значимости для MANOVA
BOX_M_ALPHA_LEVEL = 0.001  # Уровень значимости для Box's M-test

In [ ]:
# --- ЗАГРУЗКА ДАННЫХ И ПЕРВИЧНЫЙ АНАЛИЗ ---

try:
    data = pd.read_csv("../data/processed/heart_disease_uci_clustered.csv")

    data["Cluster_Hierarchical"] = data["Cluster_Hierarchical"].astype("category")
    data["sex"] = data["sex"].astype("category")

    target_features = ["age", "trestbps", "thalch", "oldpeak"]

    clean_data = data.dropna(
        subset=target_features + ["Cluster_Hierarchical", "sex"]
    ).copy()

    display(Markdown("#### **Содержание набора данных для MANOVA**"))
    display(clean_data[target_features + ["Cluster_Hierarchical", "sex"]].head())
    display(
        Markdown(
            f"Размерность данных: **{clean_data.shape[0]} строк и {len(target_features)} количественных признаков.**"
        )
    )

except FileNotFoundError as e:
    logger.error(
        "Файл `../data/processed/heart_disease_uci_clustered.csv` не найден. Убедитесь, что пайплайн кластеризации был выполнен."
    )
    raise e

## **1. Проверка статистических допущений**

* __Цель:__ Убедиться в математической обоснованности и корректности применения параметрического многомерного дисперсионного анализа (MANOVA) к выбранному набору количественных медицинских признаков.
* __Задачи:__ Доказать отсутствие избыточной линейной зависимости (мультиколлинеарности) между зависимыми переменными, подтвердить многомерную нормальность их распределения и оценить равенство (гомогенность) дисперсионно-ковариационных матриц во всех исследуемых кластерах.
* __Алгоритм использования:__
  1. __Контроль мультиколлинеарности:__ Обоснование выбора признаков на основе ранее проведенного корреляционного анализа и диагностики VIF.
  2. __Диагностика многомерной нормальности:__ Вычисление квадратов расстояний Махаланобиса для каждого наблюдения с последующей визуализацией на графике квантилей (Q-Q Plot) против теоретического распределения Хи-квадрат ($\chi^2$). Дополнительное математическое подтверждение тестом Хензе-Цирклера (Henze-Zirkler).
  3. __Проверка гомогенности ковариаций:__ Проведение Box's M-test (Критерия Бокса). В случае ожидаемого отклонения нулевой гипотезы на реальных медицинских данных — методологическое обоснование перехода к использованию наиболее устойчивого (робастного) многомерного критерия Pillai's Trace.

In [ ]:
# --- 1.1. ПРОВЕРКА МНОГОМЕРНОЙ НОРМАЛЬНОСТИ ---

# Проверка многомерной нормальности с помощью расстояния Махаланобиса и Q-Q Plot
display(
    Markdown(
        "### **Проверка статистических допущений для MANOVA**\n#### **1. Многомерная нормальность (Расстояние Махаланобиса)**"
    )
)

X = clean_data[target_features]

# Вычисление ковариационной матрицы, ее обратной и вектора средних для расчета расстояния Махаланобиса
cov_matrix = np.cov(X.T)
inv_cov_matrix = np.linalg.inv(cov_matrix)
mean_vec = np.mean(X, axis=0)

# Расчет расстояния Махаланобиса для каждой строки данных
mahalanobis_dist = [
    distance.mahalanobis(row, mean_vec, inv_cov_matrix) ** 2 for row in X.values
]

# Визуализация Q-Q Plot
display(Markdown("* **Q-Q Plot для визуальной оценки нормальности остатков**"))

fig, ax = plt.subplots(figsize=(8, 6))
# Сравнение эмпирических квантилей расстояния Махаланобиса с теоретическими квантилями χ² распределения
stats.probplot(mahalanobis_dist, dist="chi2", sparams=(len(target_features),), plot=ax)

ax.get_lines()[0].set_marker("o")
ax.get_lines()[0].set_markerfacecolor("none")
ax.get_lines()[0].set_markeredgecolor("#333333")
ax.get_lines()[0].set_alpha(0.8)
ax.get_lines()[1].set_color("black")
ax.get_lines()[1].set_linestyle("--")
ax.get_lines()[1].set_linewidth(1.5)

ax.set_title("Q-Q Plot квантилей Махаланобиса", fontsize=14, pad=15)
ax.set_xlabel(r"Теоретические квантили ($\chi^2$ распределение)", fontsize=12)
ax.set_ylabel("Квадраты расстояний Махаланобиса", fontsize=12)
plt.tight_layout()
plt.show()

# Тест Хензе-Цирклера (Mardia's / Henze-Zirkler test via pingouin)
hz_test = pg.multivariate_normality(X, alpha=ALPHA_LEVEL)
display(
    Markdown(
        f"* **Тест Хензе-Цирклера:** p-value = `{hz_test.pval:.4e}`, Нормальность: **{hz_test.normal}**"
    )
)

In [ ]:
# --- 1.2. ПРОВЕРКА ГОМАГЕННОСТИ КОВАРИАЦИОННЫХ МАТРИЦ ---

display(Markdown("#### **2. Гомогенность ковариационных матриц (Box's M-test)**"))

try:
    # Расчет Box's M-test
    box_m_result = pg.box_m(
        data=clean_data, dvs=target_features, group="Cluster_Hierarchical"
    )
    p_val_box = box_m_result.loc["box", "pval"]

    display(Markdown("* **Box's M-test**"))
    display(box_m_result)

    if p_val_box < BOX_M_ALPHA_LEVEL:
        display(
            Markdown(
                f"* **Вывод:** Box's M-test высоко значим (p-value = `{p_val_box:.4e}`). Ковариационные матрицы не равны. Однако, поскольку объемы наших кластеров относительно сбалансированы, мы методологически обоснованно переходим к использованию критерия **Pillai's Trace**, который является наиболее робастным к данному нарушению."
            )
        )
    else:
        display(
            Markdown("* **Вывод:** Гомогенность ковариационных матриц подтверждена.*")
        )

except Exception as e:
    logger.warning(f"Ошибка при расчете Box M-test: {e}")
    display(
        Markdown(
            "* **Примечание:** Ошибка при расчете Box M-test. Переходим к расчету MANOVA с использованием робастного критерия Pillai's Trace.*"
        )
    )

## **2. Однофакторный MANOVA (Оценка центроидов)**

* __Цель:__ Установить наличие статистически значимых глобальных (многомерных) различий между выделенными клиническими профилями (кластерами) по всей совокупности анализируемых количественных признаков одновременно.
* __Задачи:__ Провести многомерный дисперсионный анализ, рассчитать классические многомерные критерии (Pillai's Trace, Wilks' Lambda, Hotelling's Trace, Roy's Largest Root) и, опираясь на наиболее устойчивый к нарушениям гомогенности критерий (Pillai's Trace), принять решение об отклонении нулевой гипотезы ($H_0$) о равенстве многомерных векторов средних.
* __Алгоритм использования:__
  1. __Спецификация модели:__ Формирование математической формулы, где зависимые числовые переменные объединяются в единый многомерный вектор, а предиктором выступает категориальный фактор (принадлежность к кластеру).
  2. __Математическое моделирование:__ Обучение модели MANOVA с использованием библиотеки `statsmodels`, расчет межгрупповых и внутригрупповых матриц ковариации.
  3. __Статистический вывод:__ Извлечение и интерпретация p-value для многомерных тестов с акцентом на след Пиллая (ввиду результатов Box's M-test) и формирование итогового клинико-статистического заключения о качестве разделения сегментов.

In [ ]:
# --- 2. ОДНОФАКТОРНЫЙ MANOVA ---

display(Markdown("### **Многомерные тесты (One-Way MANOVA)**"))

# Формирование строки формулы для MANOVA
formula = f"{' + '.join(target_features)} ~ C(Cluster_Hierarchical)"
display(Markdown(f"* **Спецификация модели:** `{formula}`"))

try:
    # Инициализация и обучение модели MANOVA
    manova = MANOVA.from_formula(formula, data=clean_data)

    # Расчет многомерных тестов
    manova_results = manova.mv_test()

    # Извлечение таблицы результатов для нашего фактора (Кластера)
    cluster_results = manova_results.results["C(Cluster_Hierarchical)"]["stat"]

    display(Markdown("#### **Таблица результатов многомерных критериев**"))
    display(cluster_results)

    # Интерпретация результатов для критерия Pillai's Trace
    pillai_p_value = cluster_results.loc["Pillai's trace", "Pr > F"]
    pillai_stat = cluster_results.loc["Pillai's trace", "Value"]

    display(Markdown("#### **Клинико-статистическая интерпретация**"))
    if pillai_p_value < ALPHA_LEVEL:
        display(
            Markdown(
                f"* Значение **Pillai's trace = {pillai_stat:.3f}**, p-value = `{pillai_p_value:.4e}`.\n"
                f"* Поскольку $p < {ALPHA_LEVEL}$, **нулевая гипотеза ($H_0$) уверенно отвергается.**\n"
                f"* **Вывод:** Векторы средних (центроиды) кластеров статистически значимо различаются по уникальной комбинации исследуемых клинических признаков."
            )
        )
    else:
        display(
            Markdown(
                "* **Вывод:** Нет статистически значимых различий между векторами средних кластеров."
            )
        )

except Exception as e:
    logger.error(f"Ошибка при выполнении MANOVA: {e}")
    raise e

## **3. Апостериорный анализ (Post-Hoc MANOVA)**

* __Цель:__ Детализировать результаты многомерного теста и определить, какие именно числовые признаки вносят статистически значимый вклад в разделение кластеров, а также между какими конкретными парами групп существуют эти различия.
* __Задачи:__ Провести серию одномерных дисперсионных анализов (ANOVA) для каждой зависимой переменной с применением коррекции Бонферрони. Для признаков, показавших значимость, выполнить попарные сравнения с помощью критерия Тьюки (Tukey HSD).
* __Алгоритм использования:__
  1. __Коррекция Бонферрони:__ Расчет нового, более строгого уровня значимости $\alpha_{new} = \alpha / k$, где $k$ — количество зависимых переменных (в нашем случае 4).
  2. __Одномерные ANOVA:__ Построение моделей для каждого признака изолированно и сравнение полученных p-value с $\alpha_{new}$.
  3. __Tukey HSD:__ Проведение попарных тестов для выявления точных локализаций различий между центроидами групп.

In [ ]:
# --- 3.1. ОДНОМЕРНЫЕ ANOVA С КО РРЕКЦИЕЙ БОНФЕРРОНИ ---

display(
    Markdown("### **Одномерные дисперсионные анализы (ANOVA) с коррекцией Bonferroni**")
)

# 1. Расчет коррекции Бонферрони
k_variables = len(target_features)
alpha_bonferroni = ALPHA_LEVEL / k_variables

display(
    Markdown(
        f"* Исходный уровень значимости: $\\alpha = {ALPHA_LEVEL}$\n"
        f"* Количество проверяемых признаков: $k = {k_variables}$\n"
        f"* **Скорректированный уровень значимости (Бонферрони): $\\alpha_{{new}} = {alpha_bonferroni}$**"
    )
)

# 2. Проведение односторонних ANOVA для каждого признака
anova_results = []
significant_features = []

for var in target_features:
    # Построение модели ANOVA
    model = ols(f"{var} ~ C(Cluster_Hierarchical)", data=clean_data).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)

    # Извлечение метрик
    f_val = anova_table.loc["C(Cluster_Hierarchical)", "F"]
    p_val = anova_table.loc["C(Cluster_Hierarchical)", "PR(>F)"]

    # Проверка на статистическую значимость
    is_significant = p_val < alpha_bonferroni
    if is_significant:
        significant_features.append(var)

    anova_results.append(
        {
            "Признак": var,
            "F-статистика": f_val,
            "p-value": p_val,
            "p < α_new": is_significant,
        }
    )

# Создание DataFrame для результатов ANOVA
df_anova_results = pd.DataFrame(anova_results)
df_anova_results["p-value"] = df_anova_results["p-value"].apply(lambda x: f"{x:.4e}")
display(df_anova_results)

In [ ]:
# --- 3.2. ПОПАРНЫЕ СРАВНЕНИЯ (TUKEY HSD) ---

display(Markdown("### **Локализация различий между кластерами**"))

if not significant_features:
    display(
        Markdown(
            "*Нет признаков, прошедших порог значимости Бонферрони. Апостериорный анализ Тьюки не требуется.*"
        )
    )
else:
    for var in significant_features:
        display(Markdown(f"* **Анализ различий по признаку: `{var}`**"))

        # Проведение попарного сравнения с помощью Tukey HSD
        tukey = pairwise_tukeyhsd(
            endog=clean_data[var],
            groups=clean_data["Cluster_Hierarchical"],
            alpha=ALPHA_LEVEL,
        )

        # Конвертация результатов Tukey HSD в DataFrame
        tukey_df = pd.DataFrame(
            data=tukey._results_table.data[1:], columns=tukey._results_table.data[0]
        )

        display(tukey_df)